# Real-data validation — KLRfome (Python/JAX)

Held-out AUC on the real archaeological site data. Logic lives in
`klrfome.data.tabular`, so this notebook stays short and won't balloon.

**Workflow:** start with `FRACTION = 0.05` to confirm it works, then raise toward `1.0`.

**The key knob is `sigma`** — the default 0.5 saturates the kernel once you have many
covariates. We calibrate it from the data (median cell distance) and sweep it.


In [44]:
# Auto-reload edited klrfome modules without restarting the kernel.
# (If you still hit 'cannot import name ...', do Kernel -> Restart & Run All once.)
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys; sys.path.insert(0, str(Path.cwd().parent))
from klrfome.data.tabular import (detect_columns, detect_xy, bags_from_dataframe,
    stratified_bag_split, scale_bags, median_sigma, labels_of,
    mean_embedding_heldout, wasserstein_heldout,
    fit_mean_embedding, mean_embedding_predict, predict_xy_surface,
    site_centroids, site_ids_in, optimal_threshold,
    background_bbox, restrict_to_background_domain,
    capture_curve, reliability_curve, permutation_importance)

CSV        = Path('../site_data/r91_all_riverine_section_6_regression_data_SITENO.csv')
FRACTION   = 0.05      # <-- start small; raise toward 1.0 once it works
BAG_CAP    = 120       # max cells per bag (keeps the kernel tractable)
BACKGROUND = 'spatial' # 'spatial' = honest k-NN patches; 'random' = baseline that INFLATES AUC
RESTRICT_TO_BACKGROUND_BBOX = True  # clip sites to where background exists (see section 1b)
SEED       = 42


ImportError: cannot import name 'background_bbox' from 'klrfome.data.tabular' (/Users/mattharris/Documents/Python_local/KLRFome_JAX/klrfome/data/tabular.py)

## 1. Load & subsample


In [ ]:
df = pd.read_csv(CSV)
if FRACTION < 1.0:
    df = df.sample(frac=FRACTION, random_state=SEED)
pcol, scol, covars = detect_columns(df)
print(f'rows={len(df):,}   covariates ({len(covars)}): {covars}')
print('presence counts:', df[pcol].astype(int).value_counts().to_dict())


## 1b. Restrict to the background's spatial domain  ⚠️ (EXPLICIT, not silent)
**The background was only sampled over part of the study area** (here the upper
riverine zone, y >~ 500k). Sites outside that footprint have **no comparable
background**, so scoring them is extrapolation and the prediction surface is undefined
there. We therefore clip BOTH sites and background to the background bounding box.

This is a **known, temporary limitation of the current data**, made loud on purpose:
when full-area background becomes available, set `RESTRICT_TO_BACKGROUND_BBOX = False`
and re-run. The printout below records exactly how many sites were dropped.


In [ ]:
if RESTRICT_TO_BACKGROUND_BBOX:
    bbox = background_bbox(df)
    df, info = restrict_to_background_domain(df)
    print('=' * 64)
    print('DOMAIN RESTRICTION APPLIED  (sites clipped to background bbox)')
    print(f"  bbox: x[{bbox[0]:.0f}, {bbox[1]:.0f}]  y[{bbox[2]:.0f}, {bbox[3]:.0f}]")
    print(f"  site cells:     {info['n_site_before']:,} -> {info['n_site_after']:,}"
          f"  (dropped {info['n_site_dropped']:,} outside background domain)")
    print(f"  site locations: {info['n_siteno_before']} -> {info['n_siteno_after']}")
    print(f"  background cells in domain: {info['n_background']:,}")
    print('  -> revisit (set RESTRICT_TO_BACKGROUND_BBOX=False) when full background arrives')
    print('=' * 64)
else:
    print('NOT restricted: lower-zone sites are EXTRAPOLATION (background does not cover them)')


## 2. Build bags (sites grouped by SITENO; background as spatial k-NN patches)
`BACKGROUND='spatial'` builds each background bag as a compact neighbourhood in
(x,y) space, so it has the same within-bag spatial autocorrelation a real site does.
With `'random'`, background bags scatter across the whole region (variance ~100x a
site's) and the kernel cheats on 'compact vs scattered' -> inflated AUC. Compare the
`within-bag variance` lines below: spatial background should be the same order of
magnitude as sites; random background will be far larger.


In [ ]:
bags = bags_from_dataframe(df, bag_cap=BAG_CAP, background=BACKGROUND, seed=SEED)
y = labels_of(bags)
sizes = [b.n_samples for b in bags]
wvar  = [float(np.mean(np.var(np.asarray(b.samples), 0))) for b in bags]
print(f'bags={len(bags)}  sites={int((y==1).sum())}  background={int((y==0).sum())}')
print(f'bag size  min/median/max = {min(sizes)}/{int(np.median(sizes))}/{max(sizes)}')
print(f'within-bag variance median = {np.median(wvar):.3f}')


## 3. Train/test split + scaling (train stats only)


In [ ]:
train, test = stratified_bag_split(bags, test_fraction=0.3, seed=SEED)
train_s, test_s, mu, sd = scale_bags(train, test)
print(f'train={len(train_s)} (sites={int(labels_of(train_s).sum())})  '
      f'test={len(test_s)} (sites={int(labels_of(test_s).sum())})')


## 4. Calibrate sigma — the critical step
The histogram shows why `sigma=0.5` fails: it's far below the typical distance,
so `exp(-d^2/2sigma^2)` underflows to 0 and every bag looks identical.


In [ ]:
sig = median_sigma(train_s, seed=SEED)
print(f'calibrated sigma (median cell-cell distance) = {sig:.2f}')
print(f'kernel value at that distance with default sigma=0.5: '
      f'{np.exp(-sig**2/(2*0.5**2)):.2e}  <- saturated to ~0')
cells = np.concatenate([np.asarray(c.samples) for c in train_s])
cells = cells[np.random.default_rng(SEED).choice(len(cells), min(800,len(cells)), replace=False)]
d = np.sqrt(((cells[:,None,:]-cells[None,:,:])**2).sum(-1))[np.triu_indices(len(cells),1)]
plt.figure(figsize=(6,3)); plt.hist(d, bins=60, color='steelblue')
plt.axvline(0.5, color='r', ls='--', label='default sigma=0.5')
plt.axvline(sig, color='g', ls='--', label=f'calibrated sigma={sig:.1f}')
plt.xlabel('cell-cell distance'); plt.legend(); plt.title('Why sigma matters'); plt.show()


## 5. Mean-embedding kernel — sweep sigma


In [ ]:
sigmas = sorted({0.5} | {round(m*sig,3) for m in (0.1,0.25,0.5,1.0,1.5,2.0,3.0)})
res = []
for s in sigmas:
    auc,_,_ = mean_embedding_heldout(train_s, test_s, sigma=s, seed=SEED)
    res.append((s,auc)); print(f'  sigma={s:8.3f}   held-out AUC={auc:.4f}')
res = np.array(res)
plt.figure(figsize=(6,3)); plt.plot(res[:,0], res[:,1], 'o-')
plt.axvline(sig, color='g', ls='--', label=f'calibrated {sig:.1f}')
plt.axhline(0.5, color='k', ls=':', label='random')
plt.xlabel('sigma'); plt.ylabel('held-out AUC'); plt.legend(); plt.title('Mean-embedding AUC vs sigma'); plt.show()
best_sigma = float(res[np.argmax(res[:,1]),0]); print('best sigma =', best_sigma)


## 6. Best mean-embedding model — ROC & prediction separation


In [ ]:
from sklearn.metrics import roc_curve
auc, probs, yte = mean_embedding_heldout(train_s, test_s, sigma=best_sigma, seed=SEED)
print(f'Mean-embedding held-out AUC = {auc:.3f}  (sigma={best_sigma})')
fpr,tpr,_ = roc_curve(yte, probs)
fig,ax = plt.subplots(1,2,figsize=(10,3))
ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],'k:'); ax[0].set_title(f'ROC (AUC={auc:.3f})')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[1].hist(probs[yte==1], bins=20, alpha=.6, label='site')
ax[1].hist(probs[yte==0], bins=20, alpha=.6, label='background')
ax[1].set_title('Held-out predictions'); ax[1].legend(); plt.show()


## 7. Sliced-Wasserstein kernel (shape-aware) — the original motivation
Wasserstein compares distribution *shape*, not just the mean. Sigma is
auto-calibrated to the median Wasserstein distance.


In [ ]:
auc_w, probs_w, yte_w, sig_w = wasserstein_heldout(train_s, test_s, sigma=None,
                                                   n_projections=100, p=2, seed=SEED)
print(f'Wasserstein held-out AUC = {auc_w:.3f}  (calibrated sigma={sig_w:.3f})')
fpr,tpr,_ = roc_curve(yte_w, probs_w)
plt.figure(figsize=(5,3)); plt.plot(fpr,tpr, label=f'Wasserstein AUC={auc_w:.3f}')
plt.plot([0,1],[0,1],'k:'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.title('Wasserstein ROC'); plt.show()


## 8. Confusion matrix at the optimal threshold (held-out)
A single AUC hides operating-point behaviour. We pick the threshold that maximizes
**TSS** (Youden's J = sensitivity + specificity - 1) on the held-out bags and show
the confusion matrix there.


In [ ]:
model = fit_mean_embedding(train_s, sigma=best_sigma)
p_te  = mean_embedding_predict(model, test_s)
y_te  = labels_of(test_s)
thr, tss, (TP, FP, TN, FN) = optimal_threshold(y_te, p_te, metric='tss')
sens = TP/(TP+FN) if (TP+FN) else 0; spec = TN/(TN+FP) if (TN+FP) else 0
prec = TP/(TP+FP) if (TP+FP) else 0
print(f'optimal threshold={thr:.3f}  TSS={tss:.3f}  sens={sens:.3f}  spec={spec:.3f}  prec={prec:.3f}')
cm = np.array([[TN, FP], [FN, TP]])
fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, int(v), ha='center', va='center', fontsize=13)
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred bg', 'pred site'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['true bg', 'true site'])
ax.set_title(f'Confusion @ thr={thr:.2f}  (TSS={tss:.2f})'); plt.show()


## 9. Predicted-probability surface, with train & holdout sites overlain
The most informative diagnostic: **every** cell is scored by its own k-NN neighbourhood
patch (focal prediction; scored in batches so the full surface fits in memory).
Train sites = triangles, held-out sites = stars. Held-out stars on bright (high-P)
ground = genuine spatial generalization.

**Coverage caveat:** you can only predict where covariate cells exist. In this dataset
the background was sampled only along the river corridors (all 500k background cells
are in the upper zone), so blank areas are **no data**, not low probability. The cell
below prints the spatial coverage so you can see where the surface is/!isn't defined.
A true wall-to-wall map needs covariates on a full raster grid, not just the point CSV.


In [ ]:
pcol, scol, covars = detect_columns(df); xcol, ycol = detect_xy(df)
cell_xy = df[[xcol, ycol]].to_numpy(float)
cell_X  = (df[covars].to_numpy(float) - mu) / sd
# score EVERY cell (batched) -- complete where data exists
p_map = predict_xy_surface(model, cell_xy, cell_xy, cell_X, k=16)
print(f'scored {len(p_map):,} cells   P range [{p_map.min():.2f}, {p_map.max():.2f}]')
bg_y = df.loc[df[pcol] == 0, ycol]
print(f'background cells span y=[{bg_y.min():.0f}, {bg_y.max():.0f}] '
      f'-> surface is only defined there; sites outside it have no background context')
cent = site_centroids(df)
tr_sids, te_sids = site_ids_in(train_s), site_ids_in(test_s)
c_tr = cent[cent.index.astype(str).isin(tr_sids)]
c_te = cent[cent.index.astype(str).isin(te_sids)]
fig, ax = plt.subplots(figsize=(9, 7.5))
sc = ax.scatter(cell_xy[:, 0], cell_xy[:, 1], c=p_map, s=3, cmap='viridis',
                vmin=0, vmax=1, rasterized=True)
ax.scatter(c_tr['x'], c_tr['y'], marker='^', c='white', edgecolor='black', s=70, label='train sites')
ax.scatter(c_te['x'], c_te['y'], marker='*', c='red', edgecolor='black', s=160, label='holdout sites')
plt.colorbar(sc, label='P(site)'); ax.legend(loc='best'); ax.set_aspect('equal')
ax.set_title('Predicted probability surface (all scored cells)'); plt.show()


## 10. Gain / cumulative-capture curve
The archaeology-standard view: how much of the (modeled) landscape must you survey to
capture what fraction of sites? Cells are ranked by predicted P; the dashed line is
random. The further above the diagonal, the more efficiently the model concentrates sites.


In [ ]:
is_site = (df[pcol].to_numpy() == 1).astype(int)
area_frac, cap_frac = capture_curve(p_map, is_site)
for a in (0.10, 0.25, 0.50):
    i = max(0, int(a * len(area_frac)) - 1)
    print(f'  survey {a*100:4.0f}% of area  ->  capture {cap_frac[i]*100:4.0f}% of site-cells')
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(area_frac, cap_frac, lw=2, label='model')
ax.plot([0, 1], [0, 1], 'k--', label='random')
ax.set_xlabel('fraction of area surveyed'); ax.set_ylabel('fraction of sites captured')
ax.set_title('Cumulative capture (gain)'); ax.legend(loc='lower right'); plt.show()


## 11. Calibration (reliability) curve
Are the probabilities meaningful or just ranks? Held-out predictions are binned and
mean-predicted-P is compared to observed site frequency. On the diagonal = calibrated.
Labels show the count per bin (coarse at small FRACTION).


In [ ]:
bc, mp, of, ct = reliability_curve(p_te, y_te, n_bins=5)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.plot([0, 1], [0, 1], 'k--', label='perfect')
ax.plot(mp, of, 'o-', label='model')
for x_, y_, c_ in zip(mp, of, ct):
    ax.annotate(str(c_), (x_, y_), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('mean predicted P'); ax.set_ylabel('observed site frequency')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Reliability (held-out)'); ax.legend(loc='upper left'); plt.show()


## 12. Permutation importance
Which covariates actually drive the prediction? Each feature is shuffled across the
held-out cells; the AUC drop is its importance (averaged over repeats).


In [ ]:
base_auc, imp = permutation_importance(model, test_s, covars, n_repeats=5, seed=SEED)
order = np.argsort(imp)[::-1]
print(f'baseline held-out AUC = {base_auc:.3f}')
topn = min(15, len(covars))
names = [covars[i] for i in order[:topn]][::-1]
vals  = [imp[i] for i in order[:topn]][::-1]
fig, ax = plt.subplots(figsize=(6, 0.35 * topn + 1))
ax.barh(names, vals); ax.axvline(0, color='k', lw=0.6)
ax.set_xlabel('AUC drop when shuffled'); ax.set_title('Permutation importance (top features)'); plt.show()


## 13. Spatial residuals — which held-out sites are missed, and where
Held-out site centroids colored by their predicted P over a grey probability surface.
Red stars = missed at the optimal threshold; green = captured. This exposes spatial
structure in the errors (e.g., a whole sub-basin the model misses) that scalars hide.


In [ ]:
p_holdout = predict_xy_surface(model, c_te[['x', 'y']].to_numpy(float), cell_xy, cell_X, k=16)
miss = p_holdout < thr
print(f'held-out sites missed at thr={thr:.2f}: {int(miss.sum())}/{len(p_holdout)}')
fig, ax = plt.subplots(figsize=(9, 7.5))
ax.scatter(cell_xy[:, 0], cell_xy[:, 1], c=p_map, s=3, cmap='Greys', vmin=0, vmax=1, alpha=0.5, rasterized=True)
ss = ax.scatter(c_te['x'], c_te['y'], c=p_holdout, cmap='RdYlGn', vmin=0, vmax=1,
                s=170, marker='*', edgecolor='black')
plt.colorbar(ss, label='P(site) at holdout location'); ax.set_aspect('equal')
ax.set_title(f'Held-out site predictions (red = missed, thr={thr:.2f})'); plt.show()


## 14. Summary & next diagnostics
- Calibrated-sigma AUC > 0.5 + held-out stars on bright ground + a gain curve well
  above the diagonal = a working, honest model **within the background domain**.
- **Domain caveat (section 1b):** results apply only where background exists. Re-run
  with `RESTRICT_TO_BACKGROUND_BBOX = False` once full-area background is available.
- **Scale up:** raise `FRACTION` (0.05 -> 0.25 -> 1.0); Wasserstein is fast now (global-Q).

Still to add later:
- **Stability across seeds / spatial-CV folds** (error bars on AUC, TSS, capture).
- **Leave-one-site-out CV** as an independent honest-evaluation lens.
- **Per-covariate partial-response curves** (shape of each top feature's effect).
